# Milestone 1: Data Representation & Foundations
**Project:** Public Health Data Visualization System  
**Dataset:** Global Health Statistics (1,000,000 records)  
**Objective:** Dataset acquisition, schema definition, descriptive statistics, and initial visual outputs.

## 1. Dataset Acquisition

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Resolve paths relative to project root
PROJECT_ROOT = Path().resolve().parent if Path().resolve().name == 'notebooks' else Path().resolve()
RAW_DATA = PROJECT_ROOT / 'data' / 'raw' / 'Global Health Statistics.csv'
CHARTS_DIR = PROJECT_ROOT / 'docs' / 'charts'
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Python: {sys.version}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset path: {RAW_DATA}")
print(f"Dataset exists: {RAW_DATA.exists()}")

In [ ]:
df = pd.read_csv(RAW_DATA)
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

## 2. Data Schema & Variable Definitions

In [ ]:
schema = pd.DataFrame({
    'Column': df.columns,
    'Dtype': df.dtypes.values,
    'Non-Null Count': df.notnull().sum().values,
    'Null Count': df.isnull().sum().values,
    'Null %': (df.isnull().mean() * 100).round(2).values,
    'Sample Value': [df[col].dropna().iloc[0] if df[col].notnull().any() else 'N/A' for col in df.columns]
})
schema

In [ ]:
# Variable type classification
categorical_cols = df.select_dtypes(include='object').columns.tolist()
numerical_cols = df.select_dtypes(include='number').columns.tolist()

print(f"Categorical variables ({len(categorical_cols)}): {categorical_cols}")
print(f"Numerical variables  ({len(numerical_cols)}): {numerical_cols}")

## 3. Basic Descriptive Statistics

In [ ]:
# Central tendency, spread, and distribution shape
stats = df[numerical_cols].agg(['mean', 'median', 'std', 'var', 'min', 'max', 'skew']).T
stats.columns = ['Mean', 'Median', 'Std Dev', 'Variance', 'Min', 'Max', 'Skewness']
stats.round(3)

In [ ]:
# Categorical variable distributions
for col in ['Disease Category', 'Gender', 'Age Group', 'Treatment Type']:
    print(f"\n--- {col} ---")
    print(df[col].value_counts())

In [ ]:
# Year range and country coverage
print(f"Year range: {df['Year'].min()} – {df['Year'].max()}")
print(f"Unique countries: {df['Country'].nunique()}")
print(f"Unique diseases: {df['Disease Name'].nunique()}")
print(f"Unique disease categories: {df['Disease Category'].nunique()}")

## 4. Data Quality Assessment

In [ ]:
print("=== MISSING VALUES ===")
missing = df.isnull().sum()
missing = missing[missing > 0]
if missing.empty:
    print("No missing values detected.")
else:
    print(missing)

print("\n=== DUPLICATE ROWS ===")
print(f"Duplicate rows: {df.duplicated().sum():,}")

print("\n=== ANOMALY CHECK (Numerical Ranges) ===")
rate_cols = ['Prevalence Rate (%)', 'Incidence Rate (%)', 'Mortality Rate (%)',
             'Healthcare Access (%)', 'Recovery Rate (%)']
for col in rate_cols:
    out_of_range = df[(df[col] < 0) | (df[col] > 100)].shape[0]
    print(f"{col}: {out_of_range:,} out-of-range values")

## 5. Initial Visual Outputs

In [ ]:
sns.set_theme(style='whitegrid', palette='muted')
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Global Health Statistics — Initial Exploratory Visuals', fontsize=14, fontweight='bold')

# 1. Mortality Rate distribution
axes[0, 0].hist(df['Mortality Rate (%)'], bins=40, color='steelblue', edgecolor='white')
axes[0, 0].set_title('Distribution of Mortality Rate (%)')
axes[0, 0].set_xlabel('Mortality Rate (%)')

# 2. Disease Category counts
cat_counts = df['Disease Category'].value_counts()
axes[0, 1].barh(cat_counts.index, cat_counts.values, color='coral')
axes[0, 1].set_title('Records by Disease Category')
axes[0, 1].set_xlabel('Count')

# 3. Healthcare Access vs Recovery Rate (sampled for speed)
sample = df.sample(5000, random_state=42)
axes[1, 0].scatter(sample['Healthcare Access (%)'], sample['Recovery Rate (%)'],
                   alpha=0.3, s=10, color='seagreen')
axes[1, 0].set_title('Healthcare Access vs Recovery Rate')
axes[1, 0].set_xlabel('Healthcare Access (%)')
axes[1, 0].set_ylabel('Recovery Rate (%)')

# 4. Records per Year
year_counts = df['Year'].value_counts().sort_index()
axes[1, 1].plot(year_counts.index, year_counts.values, marker='o', color='purple', linewidth=1.5)
axes[1, 1].set_title('Records per Year')
axes[1, 1].set_xlabel('Year')
axes[1, 1].set_ylabel('Count')

plt.tight_layout()
plt.savefig(CHARTS_DIR / 'milestone1_initial_visuals.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Chart saved to {CHARTS_DIR / 'milestone1_initial_visuals.png'}")

In [ ]:
# Summary table: avg mortality by disease category
summary_table = df.groupby('Disease Category').agg(
    Records=('Country', 'count'),
    Avg_Mortality=('Mortality Rate (%)', 'mean'),
    Avg_Recovery=('Recovery Rate (%)', 'mean'),
    Avg_Healthcare_Access=('Healthcare Access (%)', 'mean')
).round(2).sort_values('Avg_Mortality', ascending=False)

print("=== Summary Table: Avg Metrics by Disease Category ===")
summary_table